# Regression Tutorial Part 2: Multi-Linear Regression

Welcome to the world of multidimensional continuous prediction. In **Multiple Linear Regression**, we model the relationship between a continuous dependent variable ($y$) and two or more independent variables ($x_1, x_2, ..., x_n$).

### The Mathematical Formulation:
$$y = \beta_0 + \beta_1x_1 + \beta_2x_2 + \dots + \beta_nx_n + \epsilon$$

Where:
* $y$ is the predicted target.
* $x_i$ are the input features.
* $\beta_0$ is the y-intercept.
* $\beta_i$ are the coefficients (weights) for each feature.
* $\epsilon$ represents the irreducible error.

### Our Analytical Pipeline:
1. **Environment Setup:** Importing data science libraries.
2. **Data Generation:** Creating a synthetic real estate dataset with 3 features.
3. **Model Training:** Using Ordinary Least Squares (OLS) to calculate the optimal $\beta$ matrix.
4. **Evaluation:** Analyzing performance using Adjusted $R^2$ to account for multiple variables.

In [ ]:
# Cell 2: Environment Setup and Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Core Scikit-Learn modules
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Set seeds and styling for analytical consistency
np.random.seed(42)
sns.set_theme(style='whitegrid')
print("Libraries imported successfully!")

## Step 1: Synthetic Data Generation (The Real Estate Market)

To deeply understand the coefficients, we will synthesize a dataset of 500 houses where we mathematically control the underlying rules.

**The Hidden Rule we want the AI to discover:**
`House Price = $50,000 + ($200 * SqFt) + ($15,000 * Bedrooms) - ($1,000 * Age) + Noise`

Notice that *Age* has a negative coefficient, meaning older houses mathematically lose value in our simulation.

In [ ]:
# Cell 4: Generating the Analytical Dataset

n_samples = 500

# 1. Generate Input Features (Independent Variables)
# Square Footage: Normal distribution around 2000 sqft
sqft = np.random.normal(2000, 500, n_samples)
# Bedrooms: Integers from 1 to 5
bedrooms = np.random.randint(1, 6, n_samples)
# Age: Uniform distribution from 0 to 50 years old
age = np.random.uniform(0, 50, n_samples)

# 2. Assemble the Feature Matrix (X)
X = pd.DataFrame({
    'Square_Feet': sqft,
    'Bedrooms': bedrooms,
    'Age_Years': age
})

# 3. Generate the Target (Dependent Variable) using our hidden mathematical rule
true_intercept = 50000
true_coeffs = np.array([200, 15000, -1000])

# Calculate base price and add Gaussian noise to simulate real-world variance
base_price = true_intercept + (X['Square_Feet'] * true_coeffs[0]) + \
             (X['Bedrooms'] * true_coeffs[1]) + (X['Age_Years'] * true_coeffs[2])
noise = np.random.normal(0, 25000, n_samples) # $25k standard deviation of noise

y = base_price + noise

# Split the dataset: 80% for training, 20% for testing unseen data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("--- Training Data Snapshot ---")
display(X_train.head())

## Step 2: Model Training and Coefficient Analysis

When we fit a Multiple Linear Regression model, the algorithm calculates the optimal $\beta$ coefficients using matrix calculus. 

The analytical power of linear regression is **interpretability**. Unlike black-box neural networks, we can look at the learned coefficients and directly understand how the model makes decisions. 
For example, if the coefficient for Bedrooms is 14,000, the model is telling us: *"Holding square footage and age constant, adding one bedroom adds $14,000 to the house value."*

In [ ]:
# Cell 6: Model Instantiation and Fitting

# Instantiate the OLS Linear Regression model
model = LinearRegression()

# Fit the model to the training data
model.fit(X_train, y_train)

# Extract and display the learned mathematical parameters
learned_intercept = model.intercept_
learned_coeffs = model.coef_

print("--- Analytical Coefficient Breakdown ---")
print(f"Base Intercept: ${learned_intercept:,.2f} (True: $50,000)")

# Map the coefficients back to their feature names using a DataFrame for clean visualization
coeff_df = pd.DataFrame( learned_coeffs, X.columns, columns=['Learned Coefficient'])
coeff_df['True Coefficient'] = true_coeffs
coeff_df['Error'] = coeff_df['Learned Coefficient'] - coeff_df['True Coefficient']

display(coeff_df)

print("\nAnalytical Observation: The algorithm successfully decoupled the variables.")
print("Even though bigger houses usually have more bedrooms, the model isolated the independent mathematical contribution of each feature!")

## Step 3: Evaluation and Adjusted R-Squared

To evaluate our model, we use Root Mean Squared Error (RMSE) to see our average dollar-error. 

Furthermore, we must introduce **Adjusted $R^2$**. Standard $R^2$ has a mathematical flaw: it will *always* increase or stay the same when you add more features, even if those features are pure random garbage. Adjusted $R^2$ penalizes the model for adding useless features, making it the superior metric for Multiple Linear Regression.

$$\text{Adjusted } R^2 = 1 - \left( \frac{(1 - R^2)(n - 1)}{n - p - 1} \right)$$
Where $n$ is the number of samples and $p$ is the number of predictors (features).

In [ ]:
# Cell 8: Making Predictions and Calculating Advanced Metrics

# Generate predictions on the unseen testing set
y_pred = model.predict(X_test)

# 1. Calculate Standard Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# 2. Calculate Adjusted R-Squared
n = len(X_test) # Number of samples in test set
p = X_test.shape[1] # Number of features (3)
adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print("--- Test Set Evaluation Metrics ---")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"Standard R-Squared:             {r2:.4f}")
print(f"Adjusted R-Squared:             {adj_r2:.4f}")

# 3. Visualizing Predictions vs Actuals
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='purple', edgecolors='k')

# Plot the "Perfect Prediction" 1:1 diagonal line
max_val = max(max(y_test), max(y_pred))
min_val = min(min(y_test), min(y_pred))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction Line')

plt.title('Actual vs. Predicted House Prices', fontsize=14)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.legend()
plt.show()

## Conclusion

You have successfully scaled from Simple to Multiple Linear Regression! 

**Key Analytical Takeaways:**
1. **Interpretability is King:** Multi-Linear Regression provides clear coefficients. You can tell stakeholders exactly how much weight the model gives to "Age" vs "Bedrooms".
2. **Matrix Math:** OLS solves for all coefficients simultaneously without needing iterative gradient descent (though gradient descent is used for massive datasets).
3. **Adjusted $R^2$:** Never rely solely on standard $R^2$ when working with multiple variables, as it creates a false sense of confidence when hoarding useless features. 

With this foundation, you are ready to tackle polynomial regression or move into classification tasks like Logistic Regression!